# Face Detection & Recognition — End-to-End Pipeline

### Real-Time Face Recognition using **YOLOv8-Face** (Detection + Tracking, fine-tuned) & **FaceNet / InceptionResnetV1** (Recognition)

**Author:** Om Choksi
**Notebook type:** End-to-end, cell-by-cell, Kaggle & Google Colab compatible
**Pipeline:** `Video/Camera → YOLOv8-Face Detector+Tracker (fine-tuned on Kaggle Face-Detection-Dataset) → Face Crop → FaceNet Embedding → Cosine Similarity Match → Identity + Temporal Voting Buffer → Annotated Output Video`

---

**What this notebook does:**
1. Sets up the environment (auto-detects Kaggle vs Colab vs local)
2. Locates the attached **Face-Detection-Dataset** (Kaggle) and fine-tunes a YOLOv8 face detector on it
3. Evaluates the fine-tuned detector (Precision, Recall, mAP50, mAP50-95, PR/F1/confusion-matrix plots)
4. Loads the fine-tuned detector + a pretrained FaceNet (`vggface2`) recognizer
5. Builds a face embedding database from your own labeled reference images
6. Runs a real-time-style recognition pipeline on a video (with an optional temporal voting buffer for stability)
7. **Evaluates recognition** with proper metrics — ROC/AUC, EER, Accuracy/Precision/Recall/F1, Confusion Matrix, similarity-score distributions — with plots
8. Summarizes every metric (detector + recognizer) in one results table

> This is a restructured, reproducible version of an original exploratory face-recognition prototype, reorganized into clean, documented, runnable sections, with a real fine-tuning + evaluation stage added on top of the Kaggle `fareselmenshawii/face-detection-dataset`.

## Table of Contents
1. [Environment Setup & Installation](#1)
2. [Imports](#2)
3. [Configuration & Path Setup (Kaggle / Colab compatible)](#3)
4. [Required Resources (weights & data)](#4)
5. [Locate & Prepare the Face-Detection Dataset (Kaggle)](#5)
6. [Fine-Tune the YOLOv8 Face Detector](#6)
7. [Detector Evaluation (Precision / Recall / mAP)](#7)
8. [Device & Model Initialization](#8)
9. [Build Face Database (Embeddings)](#9)
10. [Face Embedding Utility Function](#10)
11. [Face Recognition Pipeline (Core Function)](#11)
12. [Run the Pipeline on Video](#12)
13. [Recognition Evaluation Metrics](#13)
14. [Visualizations](#14)
15. [Results Summary](#15)
16. [Conclusion & Future Work](#16)

<a id="1"></a>
## 1. Environment Setup & Installation

Works on **Kaggle**, **Google Colab**, or a **local/GPU server**. Run this cell once per session.

In [ ]:
%%capture
# Core dependencies for detection + recognition
!pip install ultralytics -q
!pip install facenet-pytorch -q
!pip install scikit-learn matplotlib seaborn pyyaml -q


In [ ]:
# Quick GPU check (works on both Kaggle and Colab runtimes)
!nvidia-smi || echo "No GPU detected — the notebook will fall back to CPU (slower, and fine-tuning will take much longer)."


<a id="2"></a>
## 2. Imports

In [5]:
import os
import cv2
import math
import time
import json
import yaml
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import seaborn as sns
from pathlib import Path
from collections import deque, Counter, defaultdict
from itertools import combinations

import torch
from torchvision import transforms
from PIL import Image
from facenet_pytorch import InceptionResnetV1
from ultralytics import YOLO

from sklearn.metrics import (
    roc_curve, roc_auc_score, precision_recall_curve,
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, ConfusionMatrixDisplay
)

sns.set_style("whitegrid")
plt.rcParams["figure.dpi"] = 110

print("All libraries imported successfully.")


All libraries imported successfully.


<a id="3"></a>
## 3. Configuration & Path Setup (Kaggle / Colab compatible)

All tunable parameters and file paths live in **one place**. The notebook auto-detects whether it is
running on **Kaggle**, **Colab**, or **locally**, and sets sensible default directories for each — edit
`DATABASE_DIR`, `VIDEO_PATH` and `YOLO_WEIGHTS_PATH` to point at your own uploaded data/dataset.

In [22]:
def detect_environment():
    if os.path.exists("/kaggle/working"):
        return "kaggle"
    try:
        import google.colab  # noqa: F401
        return "colab"
    except ImportError:
        return "local"

ENV = detect_environment()
print(f"Detected environment: {ENV}")

if ENV == "kaggle":
    # On Kaggle, mount your dataset under /kaggle/input/<dataset-name>/...
    # and write outputs to /kaggle/working/
    BASE_DIR = "/kaggle/working"
    INPUT_DIR = "/kaggle/input"          # add your dataset here via "Add Data"
    DATABASE_DIR = f"{INPUT_DIR}/face-database/database"      # <-- update to your own identity photos dataset
    VIDEO_PATH = f"{INPUT_DIR}/face-video/subway.mp4"          # <-- update to your dataset folder
    YOLO_WEIGHTS_PATH = f"{INPUT_DIR}/models/omchoksi04/yolo8-face-weights/pytorch/default/1/yolov8n-face-keypoints.pt"  # <-- your uploaded .pt file
elif ENV == "colab":
    from google.colab import drive
    BASE_DIR = "/content"
    DATABASE_DIR = "/content/database"
    VIDEO_PATH = "/content/subway.mp4"
    YOLO_WEIGHTS_PATH = "/content/yolov8n-face-keypoints.pt"
else:
    BASE_DIR = "."
    DATABASE_DIR = "./database"
    VIDEO_PATH = "./subway.mp4"
    YOLO_WEIGHTS_PATH = "./yolov8n-face-keypoints.pt"

OUTPUT_VIDEO_PATH = f"{BASE_DIR}/result.mp4"
# Change from the keypoint model to standard detection weights
YOLO_WEIGHTS_PATH = "yolov8n.pt"

# ---- Recognition pipeline hyperparameters ----
DET_CONF_THRESHOLD = 0.6      # min YOLO detection confidence to accept a face box
SIM_DIFF_THRESHOLD = 0.7      # max (1 - cosine similarity) to accept an identity match
USE_TEMPORAL_BUFFER = True    # majority-vote over the last N crops per tracked face
BUFFER_SIZE = 5

# ---- Detector fine-tuning hyperparameters (Section 5-7) ----
FINE_TUNE_DETECTOR = True     # set False to skip fine-tuning and just use YOLO_WEIGHTS_PATH as-is
FACE_DET_EPOCHS = 20
FACE_DET_IMG_SIZE = 640
FACE_DET_BATCH = 16
FACE_DET_RUN_NAME = "face_detector_finetune"

print(f"BASE_DIR            = {BASE_DIR}")
print(f"DATABASE_DIR         = {DATABASE_DIR}")
print(f"VIDEO_PATH           = {VIDEO_PATH}")
print(f"YOLO_WEIGHTS_PATH    = {YOLO_WEIGHTS_PATH}")
print(f"OUTPUT_VIDEO_PATH    = {OUTPUT_VIDEO_PATH}")
print(f"FINE_TUNE_DETECTOR   = {FINE_TUNE_DETECTOR}")


Detected environment: kaggle
BASE_DIR            = /kaggle/working
DATABASE_DIR         = /kaggle/input/face-database/database
VIDEO_PATH           = /kaggle/input/face-video/subway.mp4
YOLO_WEIGHTS_PATH    = yolov8n.pt
OUTPUT_VIDEO_PATH    = /kaggle/working/result.mp4
FINE_TUNE_DETECTOR   = True


<a id="4"></a>
## 4. Required Resources (Weights & Data)

You need these on disk before running the pipeline cells below:

| Resource | Description | Where to get it |
|---|---|---|
| `YOLO_WEIGHTS_PATH` | A YOLOv8 face-detection **starting** checkpoint (e.g. `yolov8n-face-keypoints.pt`) — used as the base model that Section 6 fine-tunes further | Your own local `.pt` file, uploaded via Kaggle "+ Add Input" → Upload (creates a Dataset/Model), or Colab file upload |
| Face-Detection-Dataset | Labeled face bounding-box dataset used to fine-tune and evaluate the detector | [`fareselmenshawii/face-detection-dataset`](https://www.kaggle.com/datasets/fareselmenshawii/face-detection-dataset) on Kaggle — attach it via "+ Add Input" (you've already done this) |
| `DATABASE_DIR` | Folder of **your own** reference face images, one or more `.jpg` per identity, named `identity_xxx.jpg` (e.g. `om_01.jpg`) | Your own labeled photos — this is separate from the Kaggle detection dataset, which has no identity labels |
| `VIDEO_PATH` | A test video to run recognition on | Your own clip, or a public sample video |

**Important:** the Kaggle `Face-Detection-Dataset` only has bounding boxes for "is this a face" —
it has **no identity/name labels**, so it can fine-tune and evaluate the **detector** (Sections 5-7),
but it cannot be used to build the recognition database in Section 9. That still needs your own
named photos in `DATABASE_DIR`.

On **Colab**, uncomment and run the cell below to upload files interactively.
On **Kaggle**, use **"+ Add Data"** in the right-hand panel to attach a dataset/model, then update the paths above.

In [13]:
if ENV == "colab":
    from google.colab import files
    # Uncomment the lines you need:
    # print("Upload YOLO face-detection starting weights (.pt):")
    # uploaded = files.upload()
    # print("Upload a zip of your face database, then unzip it into DATABASE_DIR:")
    # uploaded = files.upload()
    # print("Upload your test video:")
    # uploaded = files.upload()
    pass
else:
    print("Not running on Colab — attach your dataset/model via Kaggle 'Add Data', "
          "or place files locally at the paths configured in Section 3.")


Not running on Colab — attach your dataset/model via Kaggle 'Add Data', or place files locally at the paths configured in Section 3.


<a id="5"></a>
## 5. Locate & Prepare the Face-Detection Dataset (Kaggle)

Your attached dataset follows the standard YOLO detection layout:

```
Face-Detection-Dataset/
├── images/
│   ├── train/
│   └── val/
├── labels/
│   ├── train/
│   └── val/
└── labels2/        <- alternate/duplicate label set shown in the Kaggle file browser
```

The cell below **auto-discovers** the dataset root under `/kaggle/input` (so you don't have to hardcode
the exact slug), then writes a `data.yaml` that `ultralytics` needs for training/evaluation.

**Note on `labels2`:** the standard structure `ultralytics` expects is `labels/train` and `labels/val`
matching `images/train` and `images/val` 1:1 by filename. The `labels2` folder is not part of that
pairing (it doesn't have `train`/`val` subfolders in your screenshot) — treat it as an alternate/backup
label set, not a hard requirement. This notebook uses `labels/train` and `labels/val` only. If your
real ground truth is actually in `labels2`, rename/restructure it to `labels/train` + `labels/val`
before running Section 6.

In [14]:
def find_face_detection_dataset(input_root="/kaggle/input"):
    '''Walk /kaggle/input and return the first directory containing images/ and labels/ subfolders.'''
    if not os.path.exists(input_root):
        return None
    for root, dirs, _ in os.walk(input_root):
        if "images" in dirs and "labels" in dirs:
            return root
    return None

if ENV == "kaggle":
    FACE_DET_DATASET_DIR = find_face_detection_dataset()
else:
    # Update this manually if running outside Kaggle (e.g. after downloading the dataset locally)
    FACE_DET_DATASET_DIR = "./face-detection-dataset"

if FACE_DET_DATASET_DIR is None:
    print("Could not auto-locate the dataset under /kaggle/input. "
          "Set FACE_DET_DATASET_DIR manually to the folder containing images/ and labels/.")
else:
    print(f"Found Face-Detection-Dataset at: {FACE_DET_DATASET_DIR}")
    train_images_dir = os.path.join(FACE_DET_DATASET_DIR, "images", "train")
    val_images_dir = os.path.join(FACE_DET_DATASET_DIR, "images", "val")
    n_train = len(os.listdir(train_images_dir)) if os.path.exists(train_images_dir) else 0
    n_val = len(os.listdir(val_images_dir)) if os.path.exists(val_images_dir) else 0
    print(f"  train images: {n_train}")
    print(f"  val images  : {n_val}")


Found Face-Detection-Dataset at: /kaggle/input/datasets/fareselmenshawii/face-detection-dataset
  train images: 13386
  val images  : 3347


In [15]:
# Build the data.yaml that ultralytics needs (single class: "face")
data_yaml_path = f"{BASE_DIR}/face_dataset.yaml"

data_yaml_content = {
    "path": FACE_DET_DATASET_DIR,
    "train": "images/train",
    "val": "images/val",
    "nc": 1,
    "names": ["face"],
}

with open(data_yaml_path, "w") as f:
    yaml.safe_dump(data_yaml_content, f, sort_keys=False)

print(f"Wrote dataset config to: {data_yaml_path}")
print(yaml.safe_dump(data_yaml_content, sort_keys=False))


Wrote dataset config to: /kaggle/working/face_dataset.yaml
path: /kaggle/input/datasets/fareselmenshawii/face-detection-dataset
train: images/train
val: images/val
nc: 1
names:
- face



<a id="6"></a>
## 6. Fine-Tune the YOLOv8 Face Detector

Starts from your existing `YOLO_WEIGHTS_PATH` checkpoint and fine-tunes it further on the Kaggle
Face-Detection-Dataset. Set `FINE_TUNE_DETECTOR = False` in Section 3 to skip this and just use
`YOLO_WEIGHTS_PATH` as-is.

In [23]:
import os
from pathlib import Path

# Dynamically find the dataset root
dataset_root = Path("/kaggle/input/datasets/fareselmenshawii/face-detection-dataset")
yaml_path = Path("/kaggle/working/face_dataset.yaml")

yaml_content = f"""\
path: {dataset_root}
train: images/train
val: images/val

names:
  0: face
"""

with open(yaml_path, "w") as file:
    file.write(yaml_content)

print(f"✅ Re-generated a clean object detection YAML at: {yaml_path}")

✅ Re-generated a clean object detection YAML at: /kaggle/working/face_dataset.yaml


In [24]:
if FINE_TUNE_DETECTOR:
    detector_model = YOLO(YOLO_WEIGHTS_PATH)

    train_results = detector_model.train(
        data=data_yaml_path,
        epochs=FACE_DET_EPOCHS,
        imgsz=FACE_DET_IMG_SIZE,
        batch=FACE_DET_BATCH,
        device=0 if torch.cuda.is_available() else "cpu",
        project=f"{BASE_DIR}/runs",
        name=FACE_DET_RUN_NAME,
        patience=10,
        exist_ok=True,
    )

    FINE_TUNED_WEIGHTS_PATH = str(Path(train_results.save_dir) / "weights" / "best.pt")
    print(f"Fine-tuning complete. Best weights saved to: {FINE_TUNED_WEIGHTS_PATH}")
else:
    detector_model = YOLO(YOLO_WEIGHTS_PATH)
    FINE_TUNED_WEIGHTS_PATH = YOLO_WEIGHTS_PATH
    print("FINE_TUNE_DETECTOR is False — using YOLO_WEIGHTS_PATH as-is, no training performed.")


Ultralytics 8.4.90 🚀 Python-3.12.13 torch-2.2.2+cu121 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/face_dataset.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dis=6.0, distill_model=None, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=20, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=face_detector_finetune, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mas

<a id="7"></a>
## 7. Detector Evaluation (Precision / Recall / mAP)

Runs `ultralytics`' built-in validation on the held-out `val` split and reports the standard object
detection metrics, plus the diagnostic plots `ultralytics` saves automatically (PR curve, F1 curve,
confusion matrix).

In [25]:
val_metrics = detector_model.val(data=data_yaml_path, split="val")

detector_precision = float(val_metrics.box.mp)      # mean precision across classes
detector_recall = float(val_metrics.box.mr)         # mean recall across classes
detector_map50 = float(val_metrics.box.map50)        # mAP @ IoU 0.50
detector_map50_95 = float(val_metrics.box.map)       # mAP @ IoU 0.50:0.95

print(f"Precision   : {detector_precision:.4f}")
print(f"Recall      : {detector_recall:.4f}")
print(f"mAP@0.50    : {detector_map50:.4f}")
print(f"mAP@0.50:0.95: {detector_map50_95:.4f}")

detector_val_save_dir = Path(val_metrics.save_dir)
print(f"Diagnostic plots saved to: {detector_val_save_dir}")


Ultralytics 8.4.90 🚀 Python-3.12.13 torch-2.2.2+cu121 CUDA:0 (Tesla T4, 14912MiB)
Model summary (fused): 73 layers, 3,005,843 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 239.7±174.0 MB/s, size: 134.1 KB)
val: Scanning /kaggle/input/datasets/fareselmenshawii/face-detection-dataset/labels/val... 3347 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 3347/3347 771.8it/s 4.3s0.1s
WARNING ⚠️ val: Cache directory /kaggle/input/datasets/fareselmenshawii/face-detection-dataset/labels is not writable, cache not saved.
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 210/210 8.3it/s 25.2s0.1ss
                   all       3347      10299        0.9      0.801      0.877       0.58
Speed: 0.7ms preprocess, 3.0ms inference, 0.0ms loss, 0.9ms postprocess per image
Results saved to /kaggle/working/runs/detect/val
Precision   : 0.9001
Recall      : 0.8006
mAP@0.50    : 0.8771
mAP@0.50:0.95: 0.5801
D

In [27]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

face_detector = YOLO(FINE_TUNED_WEIGHTS_PATH)
face_detector.to(device)

face_recognizer = InceptionResnetV1(pretrained="vggface2").eval().to(device)

print("Face detector (fine-tuned) and recognizer loaded.")


Using device: cuda:0


  0%|          | 0.00/107M [00:00<?, ?B/s]

Face detector (fine-tuned) and recognizer loaded.


<a id="9"></a>
## 9. Build Face Database (Embeddings)

Every reference image in `DATABASE_DIR` is embedded once and cached in memory as `Data_Base`.
Identity labels are parsed from the filename prefix before the first underscore
(e.g. `om_01.jpg` → identity `om`). **This is your own labeled dataset — not the Kaggle detection dataset.**

In [51]:
results_summary = pd.DataFrame({
    "Metric": [
        "Detector fine-tuned",
        "Detector epochs",
        "Detector Precision (val)",
        "Detector Recall (val)",
        "Detector mAP@0.50 (val)",
        "Detector mAP@0.50:0.95 (val)",
    ],
    "Value": [
        FINE_TUNE_DETECTOR,
        FACE_DET_EPOCHS if FINE_TUNE_DETECTOR else "n/a",
        round(detector_precision, 4),
        round(detector_recall, 4),
        round(detector_map50, 4),
        round(detector_map50_95, 4),
    ],
})
results_summary


,Metric,Value
0,Detector fine-tuned,True
1,Detector epochs,20
2,Detector Precision (val),0.9001
3,Detector Recall (val),0.8006
4,Detector mAP@0.50 (val),0.8771
5,Detector mAP@0.50:0.95 (val),0.5801


<a id="16"></a>
## 16. Conclusion & Future Work

**Summary:** This notebook implements an end-to-end face detection and recognition pipeline. The
YOLOv8-face detector is fine-tuned and evaluated on the Kaggle `fareselmenshawii/face-detection-dataset`
(Precision/Recall/mAP), and the FaceNet (VGGFace2) recognizer is evaluated with standard verification
(ROC/AUC/EER) and identification (confusion matrix) metrics on a separate, identity-labeled reference
database, plus detector/match-score diagnostics from an actual video run.

**Possible next steps:**
- Grow the Face-Detection-Dataset fine-tuning run (more epochs / larger image size) once a baseline works end-to-end
- Add face alignment (via detected keypoints) before embedding, to improve recognition robustness to pose
- Replace the leave-one-out database evaluation with a proper held-out labeled identity test set as more data becomes available
- Explore `MTCNN` or a `casia-webface`-pretrained recognizer as alternatives (see original prototype notes)
- Package the pipeline into a small CLI / API for reuse outside notebooks

---
**Notebook by Om Choksi**
GitHub: [OMCHOKSI108](https://github.com/OMCHOKSI108) · [omchoksi.codes](https://omchoksi.codes) · Medium blog